# RG3 미성형 비라벨 과다경보 원인 점검


In [ ]:
from pathlib import Path
import os,nbformat,json,joblib,numpy as np,pandas as pd
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'rg3/01_preprocess_train.ipynb').exists())
n=nbformat.read(ROOT/'rg3/01_preprocess_train.ipynb',4);exec(n.cells[1].source)
previous=Path.cwd()
try:
 os.chdir(WORK)
 setup=source('145_rg3_shortshot_unlabeled_application.ipynb')
 ns={'__name__':'__main__'};exec(setup[:setup.index('for ns in [na,nb,nd]+nc:')],ns)
finally:os.chdir(previous)
old=joblib.load(WORK/'models/reduced_shortshot15/shortshot_best.joblib')
small=joblib.load(WORK/'models/reduced_shortshot15/shortshot_best.joblib')
u=ns['u'];f=u[['TimeStamp','Barrel_Temperature_5__past10_std','Hopper_Temperature__dev10']].copy()
for key,value in [('std',ns['stdU']),('physical',ns['physU']),('shock',ns['shockU']),('joint',ns['jointU']),('context',ns['contextU'])]:
 for j,col in enumerate(old['columns'][key]):f[col]=value[:,j]
ready=np.isfinite(f[[c for cols in old['columns'].values() for c in cols]].to_numpy(float)).all(axis=1)&u.history_count.ge(10).to_numpy()&~u.sensor_conflict.to_numpy()
match=np.ones(len(u),bool)
lab=ns['lab'];normal=ns['usable']&(ns['yn']==0)
for v in ['Injection_Time','Cycle_Time','Cushion_Position','Plasticizing_Position']:
 a=lab.loc[normal,v+'_raw'].to_numpy(float);med=np.median(a);iqr=np.quantile(a,.75)-np.quantile(a,.25);tol=max(abs(med)*.1,10*iqr,.01)
 match&=u[v].between(med-tol,med+tol).to_numpy()
mask=ready&match
z=f.loc[mask].reset_index(drop=True)
p38,s38=predict_short(old,z);p15,s15=predict_short(small,z)

cuts=small['cuts']
flags=np.column_stack([s15[:,0]>=cuts['A'],s15[:,1]>=cuts['B'],(s15[:,2]>=cuts['C'])&(z.Barrel_Temperature_5__past10_std.to_numpy()>=cuts['barrel5']),(s15[:,3]>=cuts['D'])&(-z.Hopper_Temperature__dev10.to_numpy()>=cuts['hopper'])])
audit=u.loc[mask,['TimeStamp','run_id','history_count','Barrel_Temperature_5','Hopper_Temperature','Max_Injection_Pressure','Filling_Time','Cycle_Time']].reset_index(drop=True)
audit['prediction']=p15.astype(int)
for j,name in enumerate(['A','B','C','D']):
 audit[name]=flags[:,j].astype(int);audit[name+'_score']=s15[:,j]
audit.to_csv(WORK/'reports/tables/rg3_207_shortshot_alarm_diagnostics.csv',index=False,encoding='utf-8-sig')
daily=audit.groupby(audit.TimeStamp.dt.date).agg(shots=('prediction','size'),alarms=('prediction','sum'),A=('A','sum'),B=('B','sum'),C=('C','sum'),D=('D','sum'))
daily['alarm_percent']=100*daily.alarms/daily.shots
daily.to_csv(WORK/'reports/tables/rg3_207_shortshot_alarm_by_day.csv')
print('Fullfit thresholds',cuts)
print('Branches',dict(zip(['A','B','C','D'],flags.sum(axis=0).tolist())))
print('Daily',daily.to_string())
lab=ns['lab'];normal=ns['usable']&(ns['yn']==0)
stream=ns['stream']
dense=stream[stream['product'].eq('rg3')].set_index('TimeStamp').loc[lab.TimeStamp].reset_index()
assert len(dense)==591 and dense.TimeStamp.is_unique
names=ns['sensor_names'];dense_std=dense[[v+'__current_std10' for v in names]].to_numpy(float)
std_original=ns['na']['views']['std24']
profiles=[]
for j,name in enumerate(names):
 a=std_original[normal,j];b=dense_std[normal,j];query=ns['stdU'][mask,j]
 profiles.append(dict(sensor=name,sparse_training_median=float(np.nanmedian(a)),dense_training_median=float(np.nanmedian(b)),query_median=float(np.nanmedian(query))))
pd.DataFrame(profiles).to_csv(WORK/'reports/tables/rg3_207_history_feature_shift.csv',index=False)
print('Sparse versus dense medians',pd.DataFrame(profiles).to_string(index=False))
dense.to_csv(WORK/'reports/tables/rg3_207_dense_labeled_history.csv',index=False)
u.to_csv(WORK/'reports/tables/rg3_207_dense_unknown_history.csv',index=False)


